### Required libraries

In [ ]:
import pandas as pd # The correct package name is pandas, not pandans.
import numpy as np # The correct package name is numpy, not n.
import matplotlib.pyplot as plt # The correct package name is matplotlib, not m.
import seaborn as sns # The correct package name is seaborn, not s.
import sklearn as sk # scikitlearn is not a valid package name. The correct import for scikit-learn is:
import tensorflow as tf # TensorFlow is the correct package name.
import keras # Keras is the correct package name.
import zipfile # The correct package name is zipfile, not z.
import shutil # The correct package name is shutil, not sh.
import os # The correct package name is os, not o.
import pickle # The correct package name is pickle, not p.

In [13]:
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2, InceptionV3, EfficientNetB4
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

### Data Preprocessing & Augmentation

In [17]:
# Define constants
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# Data augmentation and preparation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Assuming 'fish_data' contains your dataset with category-based folders
train_generator = train_datagen.flow_from_directory(
    r'D:\MIC_code\data\train',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    r'D:\MIC_code\data\val',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# If you have a separate test set
test_generator = test_datagen.flow_from_directory(
    r'D:\MIC_code\data\test',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Get class names
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

Found 4984 images belonging to 11 classes.
Found 215 images belonging to 11 classes.
Found 3017 images belonging to 11 classes.
Classes: ['animal fish', 'animal fish bass', 'fish sea_food black_sea_sprat', 'fish sea_food gilt_head_bream', 'fish sea_food hourse_mackerel', 'fish sea_food red_mullet', 'fish sea_food red_sea_bream', 'fish sea_food sea_bass', 'fish sea_food shrimp', 'fish sea_food striped_red_mullet', 'fish sea_food trout']
Number of classes: 11


### Build CNN From Scratch

In [14]:
# Build CNN From Scratch

def create_custom_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Train custom CNN
custom_model = create_custom_cnn()
custom_history = custom_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3),
        tf.keras.callbacks.ModelCheckpoint('custom_cnn_fish.h5', save_best_only=True)
    ]
)


Epoch 1/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2304 - loss: 2.1660

155/155 ━━━━━━━━━━━━━━━━━━━━ 290s 2s/step - accuracy: 0.2819 - loss: 1.9885 - val_accuracy: 0.4271 - val_loss: 1.5974
Epoch 2/50
  1/155 ━━━━━━━━━━━━━━━━━━━━ 5:36 2s/step - accuracy: 0.4375 - loss: 1.5146

d:\MIC_code\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - accuracy: 0.4375 - loss: 1.5146 - val_accuracy: 0.4167 - val_loss: 1.5587
Epoch 3/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 46s/step - accuracy: 0.4221 - loss: 1.5850 

155/155 ━━━━━━━━━━━━━━━━━━━━ 7044s 46s/step - accuracy: 0.4596 - loss: 1.4655 - val_accuracy: 0.5729 - val_loss: 1.2938
Epoch 4/50
  1/155 ━━━━━━━━━━━━━━━━━━━━ 7:47 3s/step - accuracy: 0.5625 - loss: 1.3013

...

155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.6250 - loss: 1.3414 - val_accuracy: 0.6146 - val_loss: 1.0558
Epoch 7/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6386 - loss: 0.9884

155/155 ━━━━━━━━━━━━━━━━━━━━ 289s 2s/step - accuracy: 0.6573 - loss: 0.9151 - val_accuracy: 0.8125 - val_loss: 0.6816
Epoch 8/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.7500 - loss: 0.7745 - val_accuracy: 0.7604 - val_loss: 0.7414
Epoch 9/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7263 - loss: 0.7311

155/155 ━━━━━━━━━━━━━━━━━━━━ 261s 2s/step - accuracy: 0.7447 - loss: 0.6951 - val_accuracy: 0.8490 - val_loss: 0.4816
Epoch 10/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.7812 - loss: 0.5649 - val_accuracy: 0.8802 - val_loss: 0.4965
Epoch 11/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7971 - loss: 0.5820

155/155 ━━━━━━━━━━━━━━━━━━━━ 262s 2s/step - accuracy: 0.7851 - loss: 0.6087 - val_accuracy: 0.8542 - val_loss: 0.4676
Epoch 12/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 19s 97ms/step - accuracy: 0.7188 - loss: 0.8121 - val_accuracy: 0.8698 - val_loss: 0.4990
Epoch 13/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8301 - loss: 0.5141

155/155 ━━━━━━━━━━━━━━━━━━━━ 455s 3s/step - accuracy: 0.8328 - loss: 0.5042 - val_accuracy: 0.8906 - val_loss: 0.3438
Epoch 14/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.8750 - loss: 0.3515 - val_accuracy: 0.8490 - val_loss: 0.4554
Epoch 15/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 325s 2s/step - accuracy: 0.8564 - loss: 0.4073 - val_accuracy: 0.8490 - val_loss: 0.4376
Epoch 16/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.9062 - loss: 0.2313 - val_accuracy: 0.8333 - val_loss: 0.4767


In [23]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2, InceptionV3, EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import shutil

# Ensure models directory exists
os.makedirs('models', exist_ok=True)

DATA_DIR = r'D:\MIC_code\data'
IMG_SIZE = (224, 224)
BATCH_SIZE = 100
EPOCHS = 30

# 🔹 Data Generators (without using create_data_generators)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2   # split train/val
)

val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    color_mode="rgb",      # <-- Force RGB
    subset="training"
)

val_gen = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    color_mode="rgb",      # <-- Force RGB
    subset="validation"
)



NUM_CLASSES = len(train_gen.class_indices)
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f"Classes: {CLASS_NAMES}")

# ------------------- Models -------------------
def build_custom_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_transfer_model(base_model, input_shape, num_classes):
    base = base_model(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Define models
model_configs = {
    'CustomCNN': build_custom_cnn((*IMG_SIZE, 3), NUM_CLASSES),
    'VGG16': build_transfer_model(VGG16, (*IMG_SIZE, 3), NUM_CLASSES),
    'ResNet50': build_transfer_model(ResNet50, (*IMG_SIZE, 3), NUM_CLASSES),
    'MobileNetV2': build_transfer_model(MobileNetV2, (*IMG_SIZE, 3), NUM_CLASSES),
}

# ------------------- Training -------------------
best_acc = 0
best_model_name = ""

for name, model in model_configs.items():
    print(f"\n🚀 Training {name}...")

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        ModelCheckpoint(f'models/{name}.h5', save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.2, patience=3)
    ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    # Track best model
    val_acc = max(history.history['val_accuracy'])
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_name = name

print(f"\n✅ Best Model: {best_model_name} (Val Accuracy: {best_acc:.4f})")

# Save best model
shutil.copy(f'models/{best_model_name}.h5', 'models/best_model.h5')
print("📂 Saved as models/best_model.h5")


Found 8268 images belonging to 3 classes.
Found 2066 images belonging to 3 classes.
Classes: ['test', 'train', 'val']

🚀 Training CustomCNN...
Epoch 1/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.5476 - loss: 1.0230

83/83 ━━━━━━━━━━━━━━━━━━━━ 748s 9s/step - accuracy: 0.5910 - loss: 0.9388 - val_accuracy: 0.6026 - val_loss: 0.9374 - learning_rate: 0.0010
Epoch 2/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 734s 8s/step - accuracy: 0.6023 - loss: 0.9106 - val_accuracy: 0.6026 - val_loss: 0.9197 - learning_rate: 0.0010
Epoch 3/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 653s 8s/step - accuracy: 0.6023 - loss: 0.9068 - val_accuracy: 0.6026 - val_loss: 0.9096 - learning_rate: 0.0010
Epoch 4/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 630s 8s/step - accuracy: 0.6023 - loss: 0.9064 - val_accuracy: 0.6026 - val_loss: 0.9025 - learning_rate: 0.0010
Epoch 5/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 650s 8s/step - accuracy: 0.6023 - loss: 0.9040 - val_accuracy: 0.6026 - val_loss: 0.9023 - learning_rate: 0.0010
Epoch 6/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 640s 7s/step - accuracy: 0.6023 - loss: 0.9056 - val_accuracy: 0.6026 - val_loss: 0.9028 - learning_rate: 0.0010
Epoch 7/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 600s 7s/step - accuracy: 0.6023 - loss: 0.9042 - val_accuracy: 0.60

83/83 ━━━━━━━━━━━━━━━━━━━━ 3350s 40s/step - accuracy: 0.5825 - loss: 0.9476 - val_accuracy: 0.6026 - val_loss: 0.9275 - learning_rate: 0.0010
Epoch 2/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 3331s 40s/step - accuracy: 0.6022 - loss: 0.9108 - val_accuracy: 0.6026 - val_loss: 0.9152 - learning_rate: 0.0010
Epoch 3/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 3473s 41s/step - accuracy: 0.6021 - loss: 0.9096 - val_accuracy: 0.6026 - val_loss: 0.9347 - learning_rate: 0.0010
Epoch 4/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 3429s 41s/step - accuracy: 0.6020 - loss: 0.9077 - val_accuracy: 0.6021 - val_loss: 0.9289 - learning_rate: 0.0010
Epoch 5/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 3454s 41s/step - accuracy: 0.6020 - loss: 0.9067 - val_accuracy: 0.6026 - val_loss: 0.9404 - learning_rate: 0.0010
Epoch 6/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 6759s 82s/step - accuracy: 0.6024 - loss: 0.9039 - val_accuracy: 0.6021 - val_loss: 0.9360 - learning_rate: 2.0000e-04
Epoch 7/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 3643s 44s/step - accuracy: 0.6023 - loss: 0.8999 - 

83/83 ━━━━━━━━━━━━━━━━━━━━ 939s 11s/step - accuracy: 0.5897 - loss: 0.9411 - val_accuracy: 0.6026 - val_loss: 0.9064 - learning_rate: 0.0010
Epoch 2/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 15s/step - accuracy: 0.6037 - loss: 0.9176 

83/83 ━━━━━━━━━━━━━━━━━━━━ 1450s 18s/step - accuracy: 0.6010 - loss: 0.9193 - val_accuracy: 0.6031 - val_loss: 0.9277 - learning_rate: 0.0010
Epoch 3/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 890s 11s/step - accuracy: 0.5992 - loss: 0.9207 - val_accuracy: 0.6026 - val_loss: 0.9041 - learning_rate: 0.0010
Epoch 4/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1085s 13s/step - accuracy: 0.6017 - loss: 0.9138 - val_accuracy: 0.6026 - val_loss: 0.9065 - learning_rate: 0.0010
Epoch 5/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1109s 13s/step - accuracy: 0.6024 - loss: 0.9144 - val_accuracy: 0.6026 - val_loss: 0.9027 - learning_rate: 0.0010
Epoch 6/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1135s 13s/step - accuracy: 0.6024 - loss: 0.9076 - val_accuracy: 0.6026 - val_loss: 0.9135 - learning_rate: 0.0010
Epoch 7/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1160s 14s/step - accuracy: 0.6022 - loss: 0.9099 - val_accuracy: 0.6026 - val_loss: 0.9090 - learning_rate: 0.0010
Epoch 8/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 1218s 15s/step - accuracy: 0.6023 - loss: 0.9078 - val_a

83/83 ━━━━━━━━━━━━━━━━━━━━ 288s 3s/step - accuracy: 0.5731 - loss: 0.9917 - val_accuracy: 0.6002 - val_loss: 0.9426 - learning_rate: 0.0010
Epoch 2/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6033 - loss: 0.9181

83/83 ━━━━━━━━━━━━━━━━━━━━ 292s 4s/step - accuracy: 0.6023 - loss: 0.9149 - val_accuracy: 0.6021 - val_loss: 0.9430 - learning_rate: 0.0010
Epoch 3/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6060 - loss: 0.9030

83/83 ━━━━━━━━━━━━━━━━━━━━ 291s 3s/step - accuracy: 0.6023 - loss: 0.9065 - val_accuracy: 0.6026 - val_loss: 0.9341 - learning_rate: 0.0010
Epoch 4/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 290s 3s/step - accuracy: 0.6023 - loss: 0.9021 - val_accuracy: 0.6026 - val_loss: 0.9470 - learning_rate: 0.0010
Epoch 5/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 292s 3s/step - accuracy: 0.6024 - loss: 0.9009 - val_accuracy: 0.6026 - val_loss: 0.9592 - learning_rate: 0.0010
Epoch 6/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 288s 3s/step - accuracy: 0.6022 - loss: 0.8997 - val_accuracy: 0.6026 - val_loss: 0.9801 - learning_rate: 0.0010
Epoch 7/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 287s 3s/step - accuracy: 0.6023 - loss: 0.8991 - val_accuracy: 0.6026 - val_loss: 0.9679 - learning_rate: 2.0000e-04
Epoch 8/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 290s 3s/step - accuracy: 0.6023 - loss: 0.8952 - val_accuracy: 0.6026 - val_loss: 0.9634 - learning_rate: 2.0000e-04

✅ Best Model: ResNet50 (Val Accuracy: 0.6031)
📂 Saved as models/best_model.h5


In [ ]:
ModelCheckpoint(f'models/{name}.h5', save_best_only=True, monitor='val_accuracy')

### Transfer Learning (Pre-trained Models)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import Model

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze layers

x = layers.Flatten()(base_model.output)
x = layers.Dense(256, activation='relu')(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

resnet_model = Model(inputs=base_model.input, outputs=output)
resnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_resnet = resnet_model.fit(train_data, validation_data=val_data, epochs=20)
resnet_model.save("models/resnet_fish.h5")


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
Epoch 1/5
76/76 ━━━━━━━━━━━━━━━━━━━━ 361s 5s/step - accuracy: 0.1828 - loss: 4.5223 - val_accuracy: 0.2930 - val_loss: 2.2270
Epoch 2/5
76/76 ━━━━━━━━━━━━━━━━━━━━ 453s 6s/step - accuracy: 0.3151 - loss: 2.1124 - val_accuracy: 0.3349 - val_loss: 2.1072
Epoch 3/5
76/76 ━━━━━━━━━━━━━━━━━━━━ 357s 5s/step - accuracy: 0.3040 - loss: 2.1148 - val_accuracy: 0.3581 - val_loss: 1.9584
Epoch 4/5
76/76 ━━━━━━━━━━━━━━━━━━━━ 410s 5s/step - accuracy: 0.3304 - loss: 1.9864 - val_accuracy: 0.3209 - val_loss: 1.9255
Epoch 5/5
76/76 ━━━━━━━━━━━━━━━━━━━━ 414s 5s/step - accuracy: 0.4103 - loss: 1.6705 - val_accuracy: 0.4093 - val_loss: 1.7045
